# Week 6: Multi-Hop GraphRAG & Entity Resolution
## Project: The Corporate Brain: Architecting the Digital Employee Experience (DEX)

**Architectural Objective:** To move beyond the limitations of "Linear RAG" by implementing a **Deterministic Reasoning Substrate**. This notebook demonstrates how to solve the "Semantic Bridging" problem within the context of **Digital Employee Experience (DEX)**.

**What is DEX?**
DEX refers to the quality of an employee's interactions with the digital tools they use daily. In a high-precision enterprise, knowledge is fragmented across Slack, Jira, and GitHub. Most "Internal AI" fails DEX because it cannot bridge these silos, forcing employees to manually hunt for context. 

We are building a "Corporate Brain" that automates this discovery, turning fragmented data into a unified, navigable knowledge graph.

**The Engineering Challenge:**
1. **Entity Resolution (ER):** Mapping disparate aliases (`@sreeram`, `snudurupati`, `sreeram-dev`) to a single Global Entity ID.
2. **Multi-Hop Reasoning:** Traversing the "Intent -> Project -> Task -> Action" path using ArangoDB Query Language (AQL).
3. **Glass-Box Validation:** Providing a clear, explainable trace of how the agent synthesized its answer across domains.

**Stack:**
* **Database:** ArangoDB (Knowledge Graph)
* **Orchestration:** LangGraph (Agentic Workflow)
* **Language:** Python 3.10+


**Instructions:**
1. Ensure your ArangoDB instance is running and accessible.
2. Configure your environment variables for `ARANGO_URL`, `ARANGO_PWD`, and `OPENAI_API_KEY`.
3. Execute the cells sequentially to observe the "Identity Handshake" and the "Multi-Hop Pathfind."

In [1]:
import os
from getpass import getpass
from arango import ArangoClient
from dotenv import load_dotenv

# 1. Environment Configuration
load_dotenv()

ARANGO_URL = os.getenv("ARANGO_URL")
ARANGO_PWD = os.getenv("ARANGO_PASSWORD")
OPENAI_KEY = os.getenv("OPENAI_API_KEY")

# 2. Establish Connection
client = ArangoClient(hosts=os.environ["ARANGO_URL"])

# 2. Connect to ArangoDB
client = ArangoClient(hosts=ARANGO_URL)
db = client.db("glass_box", username="root", password=ARANGO_PWD)

print(f"Successfully connected to ArangoDB database: {db.name}")

Successfully connected to ArangoDB database: glass_box


### Architectural Deep Dive: The Schema as a Reasoning Substrate

In this cell, we have moved from simply storing data to defining a **topology**. As an AI Architect, the goal is to create a "Deterministic Reasoning Substrate" - a framework where relationships are first-class citizens.

#### Why a Formal Graph Definition Matters:
* **Integrity of the Path:** By defining the `CorporateBrainGraph`, we ensure that the AI agent can only traverse logically sound routes. It cannot "hallucinate" a connection between a `Commit` and an `Employee` directly; it must follow the chain through a `JiraTicket`.
* **Multi-Hop Efficiency:** This means looking up a Slack message, hopping to its Project, and then finding all associated Jira tickets is an index-backed operation, not a slow relational join.
* **The "Explainability" Edge:** Because the edges are named (`HAS_IDENTITY`, `REFERS_TO`), the agent can explain its reasoning in plain English: *"I found this answer because Commit X addresses Jira Ticket Y, which is part of Project Orion."*

This schema effectively bridges the **Conversational Domain** (Slack), the **Transactional Domain** (Jira), and the **Technical Domain** (GitHub) into a unified knowledge structure.

In [2]:
# 3. Refined Schema & Graph Initialization (Fixing KeyError)

# Define the collections
document_collections = [
    "Employee", "SlackUser", "JiraUser", "GithubUser", 
    "Project", "SlackMessage", "JiraTicket", "Commit"
]

edge_collections = [
    "HAS_IDENTITY", "REFERS_TO", "PART_OF", "ASSIGNED_TO", "ADDRESSES"
]

# Ensure collections exist
for coll in document_collections:
    if not db.has_collection(coll):
        db.create_collection(coll)

for coll in edge_collections:
    if not db.has_collection(coll):
        db.create_collection(coll, edge=True)

# Graph Definition
graph_name = "CorporateBrainGraph"

# Cleanup existing graph to avoid configuration mismatch
if db.has_graph(graph_name):
    db.delete_graph(graph_name)

# Professional Graph Creation
db.create_graph(
    graph_name,
    edge_definitions=[
        {
            "edge_collection": "HAS_IDENTITY",
            "from_vertex_collections": ["Employee"], # Using explicit vertex keys
            "to_vertex_collections": ["SlackUser", "JiraUser", "GithubUser"]
        },
        {
            "edge_collection": "REFERS_TO",
            "from_vertex_collections": ["SlackMessage"],
            "to_vertex_collections": ["Project"]
        },
        {
            "edge_collection": "PART_OF",
            "from_vertex_collections": ["JiraTicket"],
            "to_vertex_collections": ["Project"]
        },
        {
            "edge_collection": "ASSIGNED_TO",
            "from_vertex_collections": ["JiraTicket"],
            "to_vertex_collections": ["JiraUser"]
        },
        {
            "edge_collection": "ADDRESSES",
            "from_vertex_collections": ["Commit"],
            "to_vertex_collections": ["JiraTicket"]
        }
    ]
)

print(f"Graph '{graph_name}' initialized successfully with explicit vertex definitions.")

Graph 'CorporateBrainGraph' initialized successfully with explicit vertex definitions.


### Step 4: The Data Ingestion: Seeding the "Project Orion" Loop

Now that the substrate is ready, we need to populate it with a scenario that requires multi-hop reasoning. We aren't just dumping documents; we are creating a **linked trace** of corporate activity across three platforms.

#### The Scenario:
* **The Employee:** You (Sreeram) are the central entity.
* **The Identity Handshake:** We link your Slack, Jira, and GitHub aliases to your employee record. This is the **Entity Resolution** anchor.
* **The "Orion" Incident:** 
    1. A **Slack Message** is posted referencing "Project Orion."
    2. A **Jira Ticket** exists for the same project.
    3. A **GitHub Commit** is pushed that explicitly references the Jira Ticket.

#### Why this matters:
By the end of this cell, your database will have a path from a conversation (`Slack`) to a piece of code (`GitHub`) that the LLM can traverse deterministically.<br>
We are setting up the "Semantic Bridge" that allows the agent to answer: *"What code has Sreeram pushed for the project discussed in Slack?"*

In [3]:
# 4. Data Seeding: The "Project Orion" Multi-Domain Loop

# Clear existing data for a clean run
for coll in document_collections + edge_collections:
    db.collection(coll).truncate()

# --- 1. SEED ENTITIES ---

# The central Employee (The Anchor)
sreeram = db.collection('Employee').insert({'_key': 'EMP_99', 'name': 'Sreeram Nudurupati', 'role': 'AI Architect'})

# The Platform Aliases (The Identity Fragments)
slack_user = db.collection('SlackUser').insert({'_key': 'U_SLACK_1', 'handle': '@sreeram'})
jira_user  = db.collection('JiraUser').insert({'_key': 'U_JIRA_1', 'username': 'snudurupati'})
gh_user    = db.collection('GithubUser').insert({'_key': 'U_GH_1', 'login': 'sreeram-dev'})

# The Corporate Context
project = db.collection('Project').insert({'_key': 'PROJ_ORION', 'name': 'Project Orion'})
ticket  = db.collection('JiraTicket').insert({
    '_key': 'TICKET_402', 
    'summary': 'Optimize Memory Controller for Orion', 
    'status': 'In Progress'
})
message = db.collection('SlackMessage').insert({
    '_key': 'MSG_001', 
    'text': 'The GPU cluster for Project Orion is bottlenecking. @sreeram, can you take a look?'
})
commit  = db.collection('Commit').insert({
    '_key': 'SHA_ABC123', 
    'message': 'Initial fix for memory controller logic - fixes TICKET_402'
})

# --- 2. SEED EDGES (The Bridges) ---

# Entity Resolution: Mapping aliases to the Global ID
db.collection('HAS_IDENTITY').insert({'_from': 'Employee/EMP_99', '_to': 'SlackUser/U_SLACK_1'})
db.collection('HAS_IDENTITY').insert({'_from': 'Employee/EMP_99', '_to': 'JiraUser/U_JIRA_1'})
db.collection('HAS_IDENTITY').insert({'_from': 'Employee/EMP_99', '_to': 'GithubUser/U_GH_1'})

# Semantic Bridges: Linking domains
db.collection('REFERS_TO').insert({'_from': 'SlackMessage/MSG_001', '_to': 'Project/PROJ_ORION'})
db.collection('PART_OF').insert({'_from': 'JiraTicket/TICKET_402', '_to': 'Project/PROJ_ORION'})
db.collection('ASSIGNED_TO').insert({'_from': 'JiraTicket/TICKET_402', '_to': 'JiraUser/U_JIRA_1'})
db.collection('ADDRESSES').insert({'_from': 'Commit/SHA_ABC123', '_to': 'JiraTicket/TICKET_402'})

print("Graph seeded successfully. Multi-domain 'Project Orion' path is now live.")

Graph seeded successfully. Multi-domain 'Project Orion' path is now live.


### Step 5: The Pathfinder AQL: Executing the Semantic Bridge

With the data seeded, we now move to the "Reasoning" phase. In this step, we aren't performing a keyword search; we are executing a **Multi-Hop Traversal**.

#### What is the "Semantic Bridge"?
In a standard RAG system, "Slack" and "GitHub" are two different languages. A vector search for "Sreeram's work" might find a Slack message or a Commit, but it won't understand that the **Commit** is the fulfillment of the **Intent** expressed in the **Slack message**.

The **Semantic Bridge** is the logical path that connects these silos:
1. **The Identity Bridge:** Linking `@sreeram` (Slack) to `sreeram-dev` (GitHub) via the central `Employee` node.
2. **The Contextual Bridge:** Linking a `SlackMessage` to a `Commit` by traversing through the `Project` and `JiraTicket` nodes.

#### The "Pathfinder" Query:
We will use ArangoDB Query Language (AQL) to "walk" the graph. This allows our agent to be deterministic. Instead of guessing, it can say:
> "I found the Slack message, followed it to Project Orion, identified the assigned Jira Ticket, and verified the work via the linked GitHub Commit."

In [4]:
# 5. The Pathfinder AQL: Executing the Multi-Hop Reasoning

pathfinder_query = """
FOR slack_user IN SlackUser
    FILTER slack_user.handle == @handle
    
    // Hop 1: Find the Employee linked to this Slack handle (Identity Resolution)
    FOR employee IN 1..1 INBOUND slack_user HAS_IDENTITY
        
        // Hop 2: Find all GitHub accounts linked to this Employee
        FOR gh_user IN 1..1 OUTBOUND employee HAS_IDENTITY
            FILTER IS_SAME_COLLECTION('GithubUser', gh_user)
            
            // Hop 3: Find Jira Tickets assigned to the Jira identity of this same Employee
            FOR jira_user IN 1..1 OUTBOUND employee HAS_IDENTITY
                FILTER IS_SAME_COLLECTION('JiraUser', jira_user)
                
                FOR ticket IN 1..1 INBOUND jira_user ASSIGNED_TO
                    
                    // Hop 4: Find Commits that address those tickets (Semantic Bridge)
                    FOR commit IN 1..1 INBOUND ticket ADDRESSES
                        RETURN {
                            "employee": employee.name,
                            "slack_handle": slack_user.handle,
                            "github_login": gh_user.login,
                            "jira_ticket": ticket._key,
                            "ticket_status": ticket.status,
                            "commit_msg": commit.message,
                            "path": "Slack -> Employee -> Jira -> Commit"
                        }
"""

# Execute the multi-hop reasoning
cursor = db.aql.execute(pathfinder_query, bind_vars={'handle': '@sreeram'})
results = [doc for doc in cursor]

import json
if results:
    print("--- Multi-Hop Reasoning Result ---")
    print(json.dumps(results[0], indent=2))
else:
    print("No path found. Check your edge definitions.")

--- Multi-Hop Reasoning Result ---
{
  "employee": "Sreeram Nudurupati",
  "slack_handle": "@sreeram",
  "github_login": "sreeram-dev",
  "jira_ticket": "TICKET_402",
  "ticket_status": "In Progress",
  "commit_msg": "Initial fix for memory controller logic - fixes TICKET_402",
  "path": "Slack -> Employee -> Jira -> Commit"
}


### Step 6: Orchestrating the Reasoning Loop with LangGraph

While the AQL query we just ran is powerful, in a production "Corporate Brain," a single query is often too brittle. What if the Slack handle doesn't exist? What if the path is broken at the Jira stage? A script will just fail; an **AI Architect** builds a system that can **self-correct**.

This is where **LangGraph** comes in. We are moving from a "Script" to an **Agentic Workflow**.

#### Why LangGraph for DEX?
* **State Management:** The agent maintains a "State" (e.g., current user, identified project, found tickets) as it moves through the graph.
* **The "Glass Box" Trace:** LangGraph allows us to visualize every node transition, making the reasoning process fully auditable for your CTO.
* **Self-Correction (CRAG):** If the initial "Pathfinder" AQL returns no results, the agent doesn't give up. It can trigger a "Query Rewriter" node to try different aliases or expand the search radius in the graph.

#### Our Graph Nodes:
1.  **Identity_Resolver (ER Node):** Takes the raw query and resolves `@sreeram` to `EMP_99`.
2.  **Graph_Traverser (Reasoning Node):** Executes the multi-hop AQL to find the link between intent and action.
3.  **Status_Grader (Critic Node):** Compares the Jira status to the GitHub activity. If they mismatch, it flags a "Reality Gap."
4.  **Narrator (Output Node):** Synthesizes the final response with a full "provenance trace."

In [5]:
import operator
from typing import Annotated, List, TypedDict, Union
from langgraph.graph import StateGraph, END

# 1. Define the Agent State
class AgentState(TypedDict):
    query: str
    global_entity_id: str
    resolved_name: str
    resolved_handles: dict
    graph_results: List[dict]
    response: str
    trace: List[str]

# 2. Define the Nodes (The Reasoning Steps)

def identity_resolver(state: AgentState):
    """
    ER Node: Dynamically finds an Employee by searching all linked alias collections.
    """
    import re
    query = state['query']
    
    # Extract handle (e.g., @sreeram)
    match = re.search(r"@(\w+)|user:(\w+)", query)
    found_handle = match.group(0) if match else None
    
    if not found_handle:
        return {"trace": state.get("trace", []) + ["Node: identity_resolver -> No handle detected"]}

    # Generic AQL using UNION to search multiple alias collections properly
    er_query = """
    LET aliases = UNION(
        (FOR d IN SlackUser FILTER d.handle == @h RETURN d),
        (FOR d IN JiraUser  FILTER d.username == @h RETURN d),
        (FOR d IN GithubUser FILTER d.login == @h RETURN d)
    )
    
    FOR alias IN aliases
        FOR emp IN 1..1 INBOUND alias HAS_IDENTITY
            RETURN { id: emp._id, name: emp.name }
    """
    
    cursor = db.aql.execute(er_query, bind_vars={'h': found_handle})
    res = [d for d in cursor]

    if res:
        return {
            "global_entity_id": res[0]['id'],
            "resolved_name": res[0]['name'],
            "trace": state.get("trace", []) + [f"Node: identity_resolver -> Resolved '{found_handle}' to {res[0]['name']}"]
        }
    
    return {"trace": state.get("trace", []) + [f"Node: identity_resolver -> Found {found_handle} but no DB match"]}

def graph_traverser(state: AgentState):
    """
    Reasoning Node: Walks the graph starting from the Global ID.
    """
    emp_id = state.get("global_entity_id")
    if not emp_id:
        return {"trace": state["trace"] + ["Node: graph_traverser -> Aborted (No Entity ID)"]}
    
    # Truly Generic Pathfinding: Start at Employee, find all work.
    generic_path_query = """
    FOR emp IN Employee
        FILTER emp._id == @emp_id
        // Hop to any technical identity
        FOR identity IN 1..1 OUTBOUND emp HAS_IDENTITY
            // Hop to assigned tasks
            FOR ticket IN 1..1 INBOUND identity ASSIGNED_TO
                // Hop to evidence of work
                FOR commit IN 1..1 INBOUND ticket ADDRESSES
                    RETURN {
                        "ticket": ticket.summary,
                        "status": ticket.status,
                        "work": commit.message
                    }
    """
    cursor = db.aql.execute(generic_path_query, bind_vars={'emp_id': emp_id})
    results = [doc for doc in cursor]
    
    return {
        "graph_results": results,
        "trace": state["trace"] + [f"Node: graph_traverser -> Discovered {len(results)} work-traces for {state['resolved_name']}"]
    }

def narrator(state: AgentState):
    """
    Output Node: Synthesizes the final answer with provenance.
    """
    results = state.get("graph_results", [])
    name = state.get("resolved_name", "the user")
    
    if not results:
        res = f"I identified {name}, but I see no active work or experts linked to it in the graph."
    else:
        # Use .get() to avoid KeyErrors if we change the AQL return keys later
        reports = []
        for r in results:
            task_name = r.get('task') or r.get('ticket') or "Unknown Task"
            status = r.get('status', 'Unknown Status')
            expert = r.get('expert', name)
            reports.append(f"{expert} is working on '{task_name}' (Status: {status})")
        
        res = "DEX Status Report: " + "; ".join(reports)
    
    return {
        "response": res,
        "trace": state["trace"] + ["Node: narrator -> Final report generated"]
    }

### Step 7: Wiring the Graph: From Nodes to a Functional Brain

In this final phase of orchestration, we transform our standalone Python functions into a **stateful agent**. We use LangGraph to define the "flow of logic," explicitly setting the entry point and the sequence of execution.

#### The "Explainability" Advantage of Knowledge Graph
By wiring these nodes into a formal `StateGraph`, we gain two architectural superpowers:
1. **Deterministic Execution:** Unlike a standard LLM chain that might skip steps, this graph *forces* the agent to resolve an identity before it is allowed to query the database.
2. **Observability:** We can print the `trace` at the end of the run. For a explainability, this trace is the "receipt" that proves the AI isn't hallucinating, it is following the **semantic bridges** we built in ArangoDB.



#### The Workflow:
`START` $\rightarrow$ `identity_resolver` $\rightarrow$ `graph_traverser` $\rightarrow$ `narrator` $\rightarrow$ `END`

In [6]:
# 7. Wiring and Compiling the Corporate Brain

# Initialize the Graph
workflow = StateGraph(AgentState)

# Add our generic nodes
workflow.add_node("resolver", identity_resolver)
workflow.add_node("traverser", graph_traverser)
workflow.add_node("narrator", narrator)

# Define the edges (The Flow)
workflow.set_entry_point("resolver")
workflow.add_edge("resolver", "traverser")
workflow.add_edge("traverser", "narrator")
workflow.add_edge("narrator", END)

# Compile the executable app
corporate_brain = workflow.compile()

# --- THE INTERACTION ---

# Test Query: Using your handle to trigger the multi-hop logic
input_state = {
    "query": "What is @sreeram currently working on and what is the status?",
    "trace": []
}

print("Executing Agentic Workflow...\n")
final_output = corporate_brain.invoke(input_state)

# Display the Results
print("=== FINAL AGENT RESPONSE ===")
print(final_output['response'])

print("\n=== GLASS-BOX TRACE (Audit Log) ===")
for step in final_output['trace']:
    print(f"DEBUG: {step}")

Executing Agentic Workflow...

=== FINAL AGENT RESPONSE ===
DEX Status Report: Sreeram Nudurupati is working on 'Optimize Memory Controller for Orion' (Status: In Progress)

=== GLASS-BOX TRACE (Audit Log) ===
DEBUG: Node: identity_resolver -> Resolved '@sreeram' to Sreeram Nudurupati
DEBUG: Node: graph_traverser -> Discovered 1 work-traces for Sreeram Nudurupati
DEBUG: Node: narrator -> Final report generated


### Step 8: Multi-Entity Resolution & Contextual Linking

In this step, we move beyond simple handle lookups and implement **Contextual Resolution**. In a real-world DEX (Digital Employee Experience) scenario, users don't always ask about a person; they ask about a **problem** or a **project**.

#### The Architecture of a "Discovery" Query
When a user asks about "GPU memory issues with Project Orion," the agent must perform a more sophisticated traversal:
1. **Entity Extraction:** Identify that "Project Orion" is the primary entry point into the graph.
2. **Project-to-Task Linkage:** Traverse from the `Project` node to find active `JiraTickets` related to "Memory" or "GPU."
3. **Task-to-Owner Resolution:** Follow the `ASSIGNED_TO` edge to the `JiraUser` and bridge back to the `Employee` (the "Expert").

This allows the agent to act as an **Organizational Router**, connecting the right person to the right problem without requiring the user to know who is responsible for what.

In [7]:
# 8. Multi-Entity Resolution & Contextual Traversal

def contextual_resolver(state: AgentState):
    """
    Enhanced Resolver: Identifies Projects mentioned in the query.
    """
    query = state['query'].lower()
    trace = state.get("trace", [])
    
    # Generic AQL to find any Project whose name appears in the user's query
    project_query = """
    FOR p IN Project
        FILTER CONTAINS(@q, LOWER(p.name))
        RETURN { id: p._id, name: p.name }
    """
    
    cursor = db.aql.execute(project_query, bind_vars={'q': query})
    projects = [d for d in cursor]
    
    if projects:
        trace.append(f"Node: contextual_resolver -> Identified Project Context: {projects[0]['name']}")
        return {
            "global_entity_id": projects[0]['id'], 
            "resolved_name": projects[0]['name'],
            "trace": trace
        }
    
    return {"trace": trace + ["Node: contextual_resolver -> No project identified in query"]}

def dependency_traverser(state: AgentState):
    """
    Reasoning Node: Starting from a Project, find the 'Expert' assigned to related tasks.
    """
    proj_id = state.get("global_entity_id")
    # Safety check: ensure we are starting from a Project node
    if not proj_id or "Project" not in proj_id:
        return {"trace": state["trace"] + ["Node: dependency_traverser -> Aborted (No Project Context)"]}

    # Multi-hop: Project <-(PART_OF)- JiraTicket -(ASSIGNED_TO)-> JiraUser <-(HAS_IDENTITY)- Employee
    discovery_query = """
    FOR p IN Project
        FILTER p._id == @proj_id
        FOR ticket IN 1..1 INBOUND p PART_OF
            FOR j_user IN 1..1 OUTBOUND ticket ASSIGNED_TO
                FOR emp IN 1..1 INBOUND j_user HAS_IDENTITY
                    RETURN {
                        "expert": emp.name,
                        "task": ticket.summary,
                        "status": ticket.status
                    }
    """
    cursor = db.aql.execute(discovery_query, bind_vars={'proj_id': proj_id})
    results = [doc for doc in cursor]
    
    return {
        "graph_results": results,
        "trace": state["trace"] + [f"Node: dependency_traverser -> Identified expert {results[0]['expert'] if results else 'None'} via Jira path"]
    }

print("Contextual and Dependency nodes initialized.")

Contextual and Dependency nodes initialized.


### Step 9: Re-Wiring for Contextual Discovery

In this final orchestration, we pivot our graph's entry point. Instead of assuming we know "who" the user is talking about, we allow the agent to discover the "what" (the Project) and then let the Knowledge Graph navigate to the "who" (the Expert).

#### The Dynamic Routing Logic:
1.  **Project Discovery:** The `contextual_resolver` acts as our semantic hook.
2.  **Expert Identification:** The `dependency_finder` performs the heavy lifting, traversing three hops across silos.
3.  **Human-Readable Synthesis:** The `narrator` takes the technical path and turns it into a professional "DEX Response."

In [8]:
# 9. Wiring and Executing the Project Discovery Agent

# Define the new workflow
discovery_workflow = StateGraph(AgentState)

# Add our contextual nodes
discovery_workflow.add_node("context_resolver", contextual_resolver)
discovery_workflow.add_node("dependency_finder", dependency_traverser)
discovery_workflow.add_node("narrator", narrator)

# Set the sequence: Extract Project -> Find Expert -> Narrate
discovery_workflow.set_entry_point("context_resolver")
discovery_workflow.add_edge("context_resolver", "dependency_finder")
discovery_workflow.add_edge("dependency_finder", "narrator")
discovery_workflow.add_edge("narrator", END)

# Compile
dex_discovery_agent = discovery_workflow.compile()

# --- THE COMPLEX INTERACTION ---
complex_query = {
    "query": "Who is working on the GPU memory issues for Project Orion?",
    "trace": []
}

print("Initiating Multi-Hop Discovery...\n")
output = dex_discovery_agent.invoke(complex_query)

print(f"=== FINAL DEX RESPONSE ===\n{output['response']}")

print(f"\n=== GLASS-BOX TRACE ===")
for step in output['trace']:
    print(f"DEBUG: {step}")

Initiating Multi-Hop Discovery...

=== FINAL DEX RESPONSE ===
DEX Status Report: Sreeram Nudurupati is working on 'Optimize Memory Controller for Orion' (Status: In Progress)

=== GLASS-BOX TRACE ===
DEBUG: Node: contextual_resolver -> Identified Project Context: Project Orion
DEBUG: Node: dependency_traverser -> Identified expert Sreeram Nudurupati via Jira path
DEBUG: Node: narrator -> Final report generated


### Step 10: The Gap Detector: Identifying Stalled Work

The final hallmark of a sophisticated AI Architect is moving from **Passive Retrieval** to **Proactive Analysis**. 

In this step, we implement a **Gap Detector** node. This node doesn't just report what it finds; it looks for what is **missing**. By comparing the presence of a Jira Ticket with the absence of a linked GitHub Commit, the agent can identify "Stalled Work"—tasks that are marked as "In Progress" in management tools but show no technical movement in the codebase.

This is the ultimate "Glass Box" utility: providing a reality check between management's expectations and engineering's reality.

In [9]:
def gap_detector(state: AgentState):
    """
    Critic Node: Compares Jira intent with GitHub action.
    """
    proj_id = state.get("global_entity_id")
    trace = state["trace"]
    
    # AQL to find tickets in this project that have NO linked commits
    gap_query = """
    FOR p IN Project
        FILTER p._id == @proj_id
        FOR t IN 1..1 INBOUND p PART_OF
            // Look for inbound 'ADDRESSES' edges from Commits
            LET commits = (FOR c IN 1..1 INBOUND t ADDRESSES RETURN c)
            FILTER LENGTH(commits) == 0
            RETURN { ticket: t.summary, status: t.status }
    """
    
    cursor = db.aql.execute(gap_query, bind_vars={'proj_id': proj_id})
    gaps = [d for d in cursor]
    
    if gaps:
        gap_msg = f"WARNING: Found {len(gaps)} stalled tasks (No linked commits)."
        return {"trace": trace + [f"Node: gap_detector -> {gap_msg}"]}
    
    return {"trace": trace + ["Node: gap_detector -> No progress gaps detected. All tasks have linked activity."]}

# Final Wiring Update
final_workflow = StateGraph(AgentState)
final_workflow.add_node("context_resolver", contextual_resolver)
final_workflow.add_node("dependency_finder", dependency_traverser)
final_workflow.add_node("gap_detector", gap_detector)
final_workflow.add_node("narrator", narrator)

final_workflow.set_entry_point("context_resolver")
final_workflow.add_edge("context_resolver", "dependency_finder")
final_workflow.add_edge("dependency_finder", "gap_detector")
final_workflow.add_edge("gap_detector", "narrator")
final_workflow.add_edge("narrator", END)

final_agent = final_workflow.compile()

In [10]:
# Create a new ticket that has NO commit linked to it
new_ticket = db.collection('JiraTicket').insert({
    '_key': 'TICKET_505', 
    'summary': 'Update GPU Driver Compatibility Matrix', 
    'status': 'In Progress'
})

# Link it to Project Orion
db.collection('PART_OF').insert({'_from': 'JiraTicket/TICKET_505', '_to': 'Project/PROJ_ORION'})

# Assign it to Sreeram's Jira handle
db.collection('ASSIGNED_TO').insert({'_from': 'JiraTicket/TICKET_505', '_to': 'JiraUser/U_JIRA_1'})

print("Scenario Created: TICKET_505 is now 'In Progress' but has no linked commits.")

Scenario Created: TICKET_505 is now 'In Progress' but has no linked commits.


In [11]:
# Execute the final workflow
gap_query = {
    "query": "Give me a status update on Project Orion and check for any blockers.",
    "trace": []
}

print("Running Proactive DEX Agent...\n")
final_output = final_agent.invoke(gap_query)

print(f"=== FINAL AGENT RESPONSE ===\n{final_output['response']}")

print(f"\n=== GLASS-BOX TRACE (Audit Log) ===")
for step in final_output['trace']:
    print(f"DEBUG: {step}")

Running Proactive DEX Agent...

=== FINAL AGENT RESPONSE ===
DEX Status Report: Sreeram Nudurupati is working on 'Update GPU Driver Compatibility Matrix' (Status: In Progress); Sreeram Nudurupati is working on 'Optimize Memory Controller for Orion' (Status: In Progress)

=== GLASS-BOX TRACE (Audit Log) ===
DEBUG: Node: contextual_resolver -> Identified Project Context: Project Orion
DEBUG: Node: dependency_traverser -> Identified expert Sreeram Nudurupati via Jira path
DEBUG: Node: gap_detector -> WARNING: Found 1 stalled tasks (No linked commits).
DEBUG: Node: narrator -> Final report generated


### Summary: From Data Engineer to AI Architect (Week 6)

In this notebook, we have successfully evolved a standard RAG pattern into a **Deterministic Knowledge Graph Agent**. By moving the reasoning logic into ArangoDB (AQL) and the orchestration into LangGraph, we achieved three major milestones for the "Corporate Brain":

1.  **Semantic Bridging:** We linked unstructured Slack intent to structured Jira tasks and technical GitHub truth.
2.  **Identity Resolution:** We solved the alias problem, ensuring that `@sreeram` and `sreeram-dev` are treated as the same entity.
3.  **Proactive Auditing:** We implemented a "Gap Detector" that identifies stalled work by looking for the absence of edges—something a vector-only search cannot do.

#### Next Steps:
* **Recursive Retrieval:** Handling projects with hundreds of tickets by implementing a hierarchical "Project -> Sub-module -> Task" search.
* **Natural Language to AQL:** Replacing our hardcoded AQL templates with a dynamic "Query Generator" node using an LLM.
* **UI Integration:** Exposing this "Explainability" trace in a Streamlit dashboard for real-time organizational health monitoring.